# 01. Smallest divisor

**Tier:** Deep. **Why you'd use this:** You need the smallest capacity that still finishes under a limit. Checking every capacity from 1 upward wastes work; the feasible capacities form one contiguous range, so binary search finds the boundary.

**After this lesson you can:** binary-search the answer value; check feasibility in linear time for each mid.
About 35 minutes. **Uses:** [Binary search](../05-binary-search/01-search-insert-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb)


## Quick review

1. What are you searching over — indexes or answer values?
2. What must the feasibility check return for a candidate mid?

<details><summary>Answers</summary>

You search answer values (divisor sizes), not array indexes. The check must return whether that mid is feasible — true when the sum of ceilings stays within the threshold. Skim [Binary search](../05-binary-search/01-search-insert-lesson.ipynb) if needed.
</details>


## Recognize it

You must find the smallest integer answer — a divisor, a speed, a ship capacity — such that a linear check says "this works". The prompt mentions a threshold, a deadline, or "minimum capability".

Signals: monotone feasibility (if `mid` works, every larger guess also works), a slow linear check per guess, and an answer drawn from a numeric range, not from a sorted array of keys. A look-alike that is not this pattern: ordinary binary search for a target inside a sorted list — there the array itself is sorted, and you are not guessing a capacity.


## Why it works this way

For the smallest divisor, each number `x` contributes `ceil(x / d)` toward a total that must stay at most `threshold`. Larger `d` never increases that total, so once a divisor works, every larger divisor also works. The boolean sequence over `d = 1, 2, …, max(nums)` is therefore false for a while, then true forever. Binary search finds the first true.

The search range is the set of possible answers: low is `1`, high is `max(nums)` because a divisor that large ceilings every value to `1`. Each probe runs one linear pass over `nums` to sum the ceilings. The number of probes is about `log` of the range width, so total time grows like `n · log R` where `R` is `max(nums)`. Extra memory is a few integers.

Ceiling without floats uses integer math: `(x + d - 1) // d`. A float `math.ceil(x / d)` can round the wrong way on large integers. The loop form `while lo < hi` with `hi = mid` on success and `lo = mid + 1` on failure keeps the invariant that the answer lies in `[lo, hi]` and that `hi` is always feasible once a feasible mid has been seen.

Ordinary binary search on a sorted array looks similar in code shape but asks a different question: "is this index's value the target?". Search-on-answer asks "can we finish with this mid?". Wrong predicates that are not monotone — for example, "exactly equal to the threshold" — produce a boundary that is not meaningful, and binary search will lie. Name the monotone property before you code the check.


## When to use it

Use search-on-answer when you minimize (or maximize) a numeric capacity and a linear check decides feasibility under a monotone rule. Use ordinary binary search when the array is sorted and you need a position or presence. Use a direct formula when the answer has a closed form. Prefer scanning every candidate only when the answer range is tiny.


## Brute force, and why it's slow

Try every divisor from 1 to `max(nums)`. Count how many full feasibility scans that runs.


In [1]:
def smallest_divisor_slow(nums, threshold):
    scans = 0

    def ok(d):
        nonlocal scans
        scans += 1
        return sum((x + d - 1) // d for x in nums) <= threshold

    for d in range(1, max(nums) + 1):
        if ok(d):
            return d, scans
    return max(nums), scans


for size in (20, 40, 80):
    data = list(range(1, size + 1))
    threshold = sum(data)
    answer, scans = smallest_divisor_slow(data, threshold)
    print(size, answer, scans)


20 1 1
40 1 1
80 1 1


## Key insight

If divisor `d` is feasible, every larger divisor is too. Binary-search the first feasible `d` in `[1, max(nums)]`, and implement feasibility as one linear sum of ceilings.


## Worked example

Find the smallest divisor `d` such that the sum of `ceil(x / d)` over all `x` stays at most `threshold`.


In [2]:
def smallest_divisor(nums: list[int], threshold: int) -> int:
    # 1. Feasibility: sum of ceilings at this divisor.
    def ok(d):
        return sum((x + d - 1) // d for x in nums) <= threshold

    # 2. Answer range: 1 .. max(nums).
    lo, hi = 1, max(nums)
    # 3. Shrink to the first feasible mid.
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


print(smallest_divisor([1, 2, 5, 9], 6))
print(smallest_divisor([44, 22, 33, 11, 1], 5))


5
44


## Trace

`nums = [1, 2, 5, 9]`, `threshold = 6`:

| lo | hi | mid | sum of ceilings | ok? | next |
|---|---|---|---|---|---|
| 1 | 9 | 5 | 1+1+1+2 = 5 | yes | hi = 5 |
| 1 | 5 | 3 | 1+1+2+3 = 7 | no | lo = 4 |
| 4 | 5 | 4 | 1+1+2+3 = 7 | no | lo = 5 |
| 5 | 5 | — | — | — | answer **5** |


## Cost

About `log R` probes, each a linear scan of `n` values, so time grows like `n · log R` with `R = max(nums)`. Extra memory is constant beyond the input. That beats trying every divisor because most of the range is never probed.


## Variants

Practice ships packages under a day limit with the same search shape. Related challenges (not solved here): [Koko Eating Bananas](../../challenges/problems/46-koko-eating-bananas.ipynb) — same monotone speed check, different story.


## Common mistake

This bug is silent. On success the code sets `lo = mid` instead of `hi = mid`. When `lo` and `hi` sit next to each other and `mid` equals `lo`, the loop never shrinks and can spin — or, with a broken exit, return a divisor that was never proven minimal. A common off-by-one cousin uses `hi = mid - 1` on success and drops the only feasible answer.


In [3]:
def smallest_divisor_wrong(nums, threshold):
    def ok(d):
        return sum((x + d - 1) // d for x in nums) <= threshold

    lo, hi = 1, max(nums)
    # broken: move lo up even when mid works, skipping the minimum
    for _ in range(40):
        if lo >= hi:
            break
        mid = (lo + hi) // 2
        if ok(mid):  # noqa: SIM108 — intentional wrong branch
            lo = mid  # should be hi = mid
        else:
            lo = mid + 1
    return lo


print(smallest_divisor_wrong([1, 2, 5, 9], 6))


8


The fix sets `hi = mid` when `ok(mid)` is true, so the search keeps the feasible mid and looks for a smaller one. The worked example returns `5` for that input.


In [4]:
print(smallest_divisor([1, 2, 5, 9], 6))


5


## Python idioms

Integer ceiling `(x + d - 1) // d` avoids floats. A nested `ok` closure keeps the feasibility check next to the search. `bisect` can search a virtual range with a key function, but an explicit `lo`/`hi` loop makes the monotone shrink obvious while you learn the pattern.


In [5]:
def smallest_divisor_ceil(nums: list[int], threshold: int) -> int:
    def ok(d: int) -> bool:
        total = 0
        for x in nums:
            total += (x + d - 1) // d
        return total <= threshold

    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


print(smallest_divisor_ceil([1, 2, 5, 9], 6))


5


## Recap

- Search the answer value, not an index in a sorted array.
- Feasibility must be monotone: once true, larger guesses stay true.
- Each probe is a linear check; the number of probes is logarithmic in the answer range.

Next: [practice](01-smallest-divisor-practice.ipynb).
